In [1]:

import matplotlib.pyplot as plt
from astropy.wcs import WCS
from astropy.coordinates import SkyCoord
from astroquery.hips2fits import hips2fits
import numpy as np
import warnings
from matplotlib.colors import Normalize
import matplotlib.pyplot as plt
from reproject import reproject_interp
from astropy.io import fits
import astropy.units as u
import plotfancy as pf
pf.housestyle_rcparams()
import os
from glob import glob
from sdss_stripe import parse_stripes, plot_wrapped
import esutil.coords as coords
import os
from shapely.geometry import Polygon
from shapely.ops import unary_union

In [2]:
input_hdu = hips2fits.query(
            hips='CDS/P/DM/I/350/gaiaedr3',
            width=4096,
            height=2048,
            ra=90 * u.deg,
            dec=90 * u.deg,
            fov=360 * u.deg,
            projection='MOL',
            coordsys='icrs',
            format='fits'
        )

ReadTimeout: HTTPSConnectionPool(host='alasky.cds.unistra.fr', port=443): Read timed out. (read timeout=30)

In [ ]:
hdu = input_hdu[0]
wcs_gaia = WCS(hdu.header)
image_data = hdu.data.copy() 
image_data[np.isnan(image_data)] = 0  
with np.errstate(divide='ignore'):
    plot_data = np.log1p(image_data)  

vmin = np.percentile(plot_data[plot_data > 0], 1) 
vmax = np.percentile(plot_data, 99.5) 
norm = Normalize(vmin=vmin, vmax=vmax)

ny, nx = plot_data.shape  
pix_x, pix_y = np.meshgrid(np.arange(nx), np.arange(ny))   
world_coords = wcs_gaia.pixel_to_world(pix_x, pix_y) 
lon = world_coords.ra.wrap_at(180 * u.deg).radian
lat = world_coords.dec.radian

finite_mask = np.isfinite(lon) & np.isfinite(lat) 
lon = lon[finite_mask] 
lat = lat[finite_mask] 
plot_data_filtered = plot_data[finite_mask]  

# fig = plt.figure(figsize=(8, 4)) 
# ax = fig.add_subplot(111, projection='mollweide')  
# im = ax.scatter(lon, lat, c=plot_data_filtered, cmap='hot', norm=norm, s=0.1)  
# ax.grid(True, color='white', ls='solid', alpha=0.7)  


In [ ]:
search_path = os.path.join('/Volumes/Expansion/exp_thardy/cubes/', '**', '*.fits*')
cube_files = glob(search_path, recursive=True)

search_path2 = os.path.join('/Volumes/Expansion/exp_thardy/cubes_new/', '**', '*.fits*')
cube_files2 = glob(search_path2, recursive=True)
for g in cube_files2:
    cube_files.append(g)

print(f"Found {len(cube_files)} FITS files to process.")
all_footprints_deg = []
all_centers_deg = []

for i, f in enumerate(cube_files):
    try:
        with fits.open(f) as hdul:
            wcs_header = None
            for hdu in hdul:
                if (hdu.header.get('CTYPE1') and hdu.header.get('CTYPE2') and
                    hdu.header.get('NAXIS', 0) >= 2):
                    wcs_header = hdu.header
                    break
            if wcs_header is None:
                raise ValueError("No valid WCS found.")

            wcs = WCS(wcs_header)
            footprint = wcs.celestial.calc_footprint()
            center = wcs.celestial.wcs.crval
            all_footprints_deg.append(footprint)
            all_centers_deg.append(center)
    except Exception as e:
        print(f"Warning: Could not process file {os.path.basename(f)}. Reason: {e}")
        continue

if not all_centers_deg:
    print("Error: Could not extract valid WCS information from any FITS files.")

# --- 2. Create the all-sky plot ---
print("\nGenerating all-sky plot...")
# fig = plt.figure(figsize=(12, 7))
# ax = fig.add_subplot(111, projection="mollweide")

# # --- Plot markers and footprints ---
# # Plot a visible marker for the center of each field
# for center_deg in all_centers_deg:
#     # Convert RA to radians in the range [-pi, pi] for Mollweide
#     ra_rad = np.deg2rad(center_deg[0])
#     ra_rad = ra_rad if ra_rad <= np.pi else ra_rad - 2 * np.pi
#     dec_rad = np.deg2rad(center_deg[1])
#     ax.plot(ra_rad, dec_rad, 'o', color='red', markersize=5, alpha=0.8)

all_centers_deg = np.array(all_centers_deg)
coord = SkyCoord(all_centers_deg, unit=u.deg)

In [ ]:
# fig = plt.figure(figsize=(5, 4))

fig, ax1 = plt.subplots(subplot_kw=dict(projection='mollweide'), figsize=(5,3)) 
# fig, ax1 = plt.subplots(1) 

im = ax1.scatter(lon, lat, c=plot_data_filtered, cmap='magma', norm=norm, s=0.1)  
ax1.scatter(coord.ra.wrap_at(180 * u.deg).radian, coord.dec.radian, s=50, marker='x', zorder=100, color='white', label='This Work')

correct_ylabels = []
# Loop through the original tick objects
for tick in ax1.get_yticklabels():
    original_text = tick.get_text()
    # Check if the string ends with the degree symbol
    if original_text.endswith('°'):
        # Replace the degree symbol with the LaTeX representation
        new_text = original_text[:-1] + r'$^{\circ}$'
        correct_ylabels.append(new_text)
    else:
        # If no degree symbol, add the original text
        correct_ylabels.append(original_text)
ax1.set_yticklabels(correct_ylabels)
# tick_hours = [r'22$^h$', ,  r'12$^h$', r'2$^h$']
tick_hours =  [ r'14$^h$',r'16$^h$',r'18$^h$',r'20$^h$',r'22$^h$', r'0$^h$', r'2$^h$', r'4$^h$', r'6$^h$',r'8$^h$', r'10$^h$']
ax1.set_xticklabels(tick_hours, color='white')

# ax1.coords[0].set_separator(('d', "'", '"'))

##### OTHER FIELDS 

# GOODS-S Field (approx. 10' x 16')
ra_center_gs = 53.125*u.deg
dec_center_gs = -27.805*u.deg
plt.scatter(ra_center_gs.to(u.radian), dec_center_gs.to(u.radian), s=30, color='#77aca2', marker='^', label='MW-GOOD-S', zorder=100)

# COSMOS Field (approx. 1.4 deg x 1.4 deg)
ra_center_cosmos = 150.117*u.deg
dec_center_cosmos = 2.206*u.deg
plt.scatter(ra_center_cosmos.to(u.radian), dec_center_cosmos.to(u.radian), s=120, color='#77aca2', marker='v', label='MW-COSMOS', zorder=100)

main_survey_polygons = []
southern_stripes = []
stripes = parse_stripes('sdss_stripe.par')
for stripe in stripes:
    # Separate the main survey from the southern stripes based on key
    if 'eta' in stripe:
        eta = stripe['eta']
        lambda_min = stripe['lambdaMin']
        lambda_max = stripe['lambdaMax']
        eta_top = eta + 1.25
        eta_bottom = eta - 1.25
        
        main_survey_polygons.append(Polygon([
            (lambda_min, eta_bottom),
            (lambda_max, eta_bottom),
            (lambda_max, eta_top),
            (lambda_min, eta_top)
        ]))
    else:
        southern_stripes.append(stripe)

# # Merge the main survey polygons into a single shape
# if main_survey_polygons:
#     merged_survey = unary_union(main_survey_polygons)
#     boundary_lambda, boundary_eta = merged_survey.exterior.xy
#     ra_boundary, dec_boundary = coords.sdss2eq(np.array(boundary_lambda), np.array(boundary_eta))
#     plot_wrapped(ax1, ra_boundary, dec_boundary, color='#ff004f', linewidth=2, label='SDSS-N', zorder=10)


ax1.legend(fontsize=9, loc='lower right')

ax1.grid(True, color='white', ls='solid', alpha=0.3)  
ax1.set_xlabel('Right Ascenscion')
ax1.set_ylabel('Declination')
fig.savefig(f'figs/allsky.png', dpi=600, bbox_inches='tight')